# Defense systems on the "payload-free" plasmidome

`cryptic_plasmids.ipynb` defined **payload-free** as carrying no AMR (`card_n_arg`), no virulence
(`plasann_n_virulence`) and no metal/biocide resistance (`plasann_n_metal_biocide`) — 71,414
small plasmids, half the plasmidome.

**Defense systems were never in that list.** DefenseFinder finds **3,647 of them on 3,362 of these
plasmids**, dominated by restriction-modification. A plasmid carrying a complete Type II RM system
is still "payload-free" under the definition we have been using.

That is a gap in the definition rather than a contradiction of it, and it is worth a sentence in
any write-up that uses the term.

Two things temper the headline, and both are worked through below rather than left as caveats:

1. **Carriage tracks gene count over a 24× range.** A defense system needs genes to be detected, so
   "5.1% of plasmids carry one" is substantially a statement about plasmid size.
2. **Roughly 44% of calls rest on a single gene.** For `RM_Type_IIG`, `AbiQ`, `HEC-09` and
   `PD-T4-5` that is the correct architecture; for others it means the system is incomplete.

Upstream: `dark_plasmidome.ipynb` (what the dark proteome is), `antidefense.ipynb` (the anti-defense
question, where this same run's AntiDefenseFinder half is analysed). This notebook reads finished
outputs and runs no searches.

## 1. What DefenseFinder found

DefenseFinder 3.0.0 was run on **whole plasmids in gene order** (MacSyFinder `gembase`), not on
clustered family representatives — MacSyFinder needs genomic context to call a system at all. That
is the key methodological difference from the Pfam and dbAPIS screens elsewhere in this project,
which are pure homology.

In [ ]:
# Defense systems on the payload-free small plasmidome — setup.
# Reads the finished DefenseFinder run only; nothing here re-runs MacSyFinder.
#   scripts/build_defensefinder_input.py -> gembase input (whole plasmids, gene order)
#   scripts/defensefinder_array.sbatch   -> data/defensefinder_run/out/  (33-task array)
#   scripts/summarize_defensefinder.py   -> df_systems.tsv, df_genes.tsv
import importlib.util
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd()
while not (ROOT / "data" / "defensefinder_run").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
R, D = ROOT / "data/defensefinder_run", ROOT / "data/dark_orf_run"
PP = ROOT / "data/plasmidscope_primary"

S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID, FAINT = "#0b0b0b", "#52514e", "#dcdbd4", "#d4d3cc"

def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)

sysd = pd.read_csv(R / "df_systems.tsv", sep="\t")
gns = pd.read_csv(R / "df_genes.tsv", sep="\t")
idmap = pd.read_csv(R / "plasmid_id_map.tsv", sep="\t")
SEARCHED = set(idmap["plasmid_id"])

meta = pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False,
                   usecols=["plasmid_id", "size_bp", "gc_percent", "plasann_n_cds",
                            "mob_mobility", "mob_cluster", "mob_rep_types", "hab_sub",
                            "is_clinical"])
m = meta[meta.plasmid_id.isin(SEARCHED)].copy()
m["carrier"] = m.plasmid_id.isin(set(sysd["plasmid_id"]))

# One Health compartments from hab_sub + is_clinical — never hab_top. Map imported from the
# Phase-3 script so this notebook cannot drift from it.
spec = importlib.util.spec_from_file_location("oh", ROOT / "scripts/analyze_amr_onehealth_mobility.py")
oh = importlib.util.module_from_spec(spec)
spec.loader.exec_module(oh)

def compartment(r):
    if r["hab_sub"] in oh.HUMAN_SUBS:
        return "Human - clinical" if r["is_clinical"] == 1 else "Human - community"
    v = oh.COMPARTMENT_MAP.get(r["hab_sub"])
    return v[1] if isinstance(v, tuple) else v
m["compartment"] = m.apply(compartment, axis=1)

# gembase gene id -> dark / named, so we can measure the contribution to the dark proteome
by = defaultdict(list)
for line in open(D / "all_cds.faa"):
    if line.startswith(">"):
        pid, idx, kind = line[1:].strip().rsplit("|", 2)
        by[pid].append((int(idx), kind))
p2g = dict(zip(idmap["plasmid_id"], idmap["gembase_id"]))
lookup = {}
for pid, lst in by.items():
    for i, (idx, kind) in enumerate(sorted(lst), 1):
        lookup[f"{p2g[pid]}_{i:05d}"] = kind
gns["activity"] = gns["sys_id"].map(dict(zip(sysd["sys_id"], sysd["activity"])))
gns["kind"] = gns["hit_id"].map(lookup)

N_SEARCHED, N_DARK = len(m), 378_552
print(f"plasmids searched      {N_SEARCHED:,}  (payload-free small plasmids with PlasAnn genes)")
print(f"systems called         {len(sysd):,} on {sysd['plasmid_id'].nunique():,} plasmids "
      f"({sysd['plasmid_id'].nunique() / N_SEARCHED * 100:.2f}%)")
print(f"  Defense              {(sysd.activity == 'Defense').sum():,}")
print(f"  Antidefense          {(sysd.activity == 'Antidefense').sum():,}  "
      f"(analysed in antidefense.ipynb — largely false positives)")
print(f"genes assigned         {len(gns):,}")

In [ ]:
# §1 — what was found. Evidence quality is split out because MacSyFinder calls some system types
# from a single gene by design (RM_Type_IIG is one fused protein; AbiQ, HEC-09, PD-T4-5 are
# single-gene systems), while for others a single-gene call means the system is incomplete.
DEF = sysd[sysd.activity.eq("Defense")].copy()
DEFTYPE = (DEF.groupby("type")
           .agg(systems=("sys_id", "size"), plasmids=("plasmid_id", "nunique"),
                multi=("genes_count", lambda x: (x > 1).sum()))
           .sort_values("systems", ascending=False))
DEFTYPE["single"] = DEFTYPE["systems"] - DEFTYPE["multi"]

print(f"DEFENSE: {len(DEF):,} systems on {DEF['plasmid_id'].nunique():,} plasmids "
      f"({DEF['plasmid_id'].nunique() / N_SEARCHED * 100:.2f}% of those searched)\n")
print(DEFTYPE.head(16).to_string())

multi = DEF[DEF.genes_count > 1]
print(f"\nevidence: {(DEF.genes_count == 1).sum():,}/{len(DEF):,} "
      f"({(DEF.genes_count == 1).mean() * 100:.1f}%) single-gene | "
      f"{len(multi):,} multi-gene on {multi['plasmid_id'].nunique():,} plasmids "
      f"({multi['plasmid_id'].nunique() / N_SEARCHED * 100:.2f}%)")

RM = DEF[DEF.type.str.startswith("RM_")]
print(f"\nrestriction-modification alone: {len(RM):,} systems ({len(RM) / len(DEF) * 100:.0f}% "
      f"of all defense) on {RM['plasmid_id'].nunique():,} plasmids")
print("\nby broad class:")
cls = pd.Series({
    "restriction-modification": len(RM),
    "toxin-antitoxin / abortive infection":
        int(DEF.type.isin(["MazEF", "AbiQ", "AbiD", "AbiL", "AbiZ", "RnlAB", "AbiE",
                           "AbiH", "AbiO", "AbiU"]).sum()),
    "signalling (CBASS / Thoeris / Pycsar)":
        int(DEF.type.isin(["CBASS", "Thoeris", "Pycsar"]).sum()),
    "other / uncharacterised class": 0})
cls["other / uncharacterised class"] = len(DEF) - cls.iloc[:3].sum()
print((pd.DataFrame({"systems": cls, "%": cls / len(DEF) * 100}).round(1)).to_string())

In [ ]:
# Figure 1 — the defense repertoire. Blue = system supported by co-located genes;
# grey = single-gene call (correct architecture for some types, homology-only for others).
fig, ax = plt.subplots(figsize=(10.5, 6.4))
t = DEFTYPE.head(16).iloc[::-1]
y = np.arange(len(t))
ax.barh(y, t["multi"], height=0.46, color=S1, label="multi-gene (co-located)")
ax.barh(y, t["single"], left=t["multi"], height=0.46, color=NEUT, label="single-gene call")
for i, (v, pl) in enumerate(zip(t["systems"], t["plasmids"])):
    ax.text(v + t["systems"].max() * 0.015, i, f"{v:,}  ({pl:,} plasmids)",
            va="center", fontsize=8, color=INK2)
ax.set(yticks=y, yticklabels=t.index, xlim=(0, t["systems"].max() * 1.30))
ax.tick_params(axis="y", labelsize=9)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_xlabel("systems called", fontsize=9, color=INK2)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="lower right")
ax.set_title(f"Figure 1 · Defense systems on \"payload-free\" plasmids\n"
             f"{len(DEF):,} systems on {DEF['plasmid_id'].nunique():,} of {N_SEARCHED:,} plasmids "
             f"— restriction-modification is {len(RM) / len(DEF) * 100:.0f}% of everything found",
             fontsize=11, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()

## 2. The confound that governs everything else

Before any prevalence claim: a defense system is only detectable if the plasmid has the genes to
carry it. Carriage should therefore rise with gene count — and it does, steeply. This is stated
here rather than buried, because it caps how strongly any of the numbers below can be read.

In [ ]:
# §2 — the confound that governs everything below. A defense system needs genes to be detected,
# so carriage is partly a statement about how many genes a plasmid has. Quantify it before
# reporting any prevalence.
m["cds_bin"] = pd.cut(m.plasann_n_cds, [0, 3, 5, 8, 12, 20, 10**6],
                      labels=["1-3", "4-5", "6-8", "9-12", "13-20", ">20"])
CDSTAB = (m.groupby("cds_bin", observed=True)
          .agg(plasmids=("plasmid_id", "size"), carriers=("carrier", "sum")))
CDSTAB["rate %"] = CDSTAB["carriers"] / CDSTAB["plasmids"] * 100
print("carriage rate by CDS count:")
print(CDSTAB.round(2).to_string())
lo, hi = CDSTAB["rate %"].iloc[0], CDSTAB["rate %"].iloc[-1]
print(f"\n  {hi / lo:.0f}x gradient across the range. The headline "
      f"'{m.carrier.mean() * 100:.1f}% of plasmids carry a defense system' is\n"
      f"  therefore not a clean number — it is largely a statement about gene content.\n")

print("carrier vs non-carrier:")
cmp_ = (m.groupby("carrier")
        .agg(plasmids=("plasmid_id", "size"), median_bp=("size_bp", "median"),
             median_CDS=("plasann_n_cds", "median"), median_GC=("gc_percent", "median")))
print(cmp_.round(1).to_string())
u, p = stats.mannwhitneyu(m.loc[m.carrier, "size_bp"], m.loc[~m.carrier, "size_bp"])
rbc = 2 * u / (m.carrier.sum() * (~m.carrier).sum()) - 1
print(f"\nsize: Mann-Whitney U p = {p:.3g}, rank-biserial = {rbc:+.3f} "
      f"(carriers are {cmp_.loc[True, 'median_bp'] / cmp_.loc[False, 'median_bp']:.1f}x larger)")
print("\nmobility (% within group):")
print((pd.crosstab(m.carrier, m.mob_mobility, normalize="index") * 100).round(1).to_string())
print(f"\nreplicon typed (mob_rep_types): "
      f"carriers {m.loc[m.carrier, 'mob_rep_types'].notna().mean() * 100:.1f}% vs "
      f"non-carriers {m.loc[~m.carrier, 'mob_rep_types'].notna().mean() * 100:.1f}%")

In [ ]:
# Figure 2 — the size confound, stated plainly rather than buried in a caveat.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.1, 1]})

x = np.arange(len(CDSTAB))
axA.bar(x, CDSTAB["rate %"], width=0.5, color=S1)
for i, (r, n) in enumerate(zip(CDSTAB["rate %"], CDSTAB["plasmids"])):
    axA.text(i, r + CDSTAB["rate %"].max() * 0.03, f"{r:.1f}%\nn={n:,}",
             ha="center", fontsize=8, color=INK2)
axA.set(xticks=x, xticklabels=CDSTAB.index, yticks=[],
        ylim=(0, CDSTAB["rate %"].max() * 1.30))
axA.set_xlabel("CDS on the plasmid", fontsize=9, color=INK2)
axA.set_title(f"A · Carriage tracks gene count — a {hi / lo:.0f}× gradient\n"
              f"the overall {m.carrier.mean() * 100:.1f}% is mostly this, not biology",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

parts = [m.loc[~m.carrier, "size_bp"], m.loc[m.carrier, "size_bp"]]
bp = axB.boxplot(parts, positions=[0, 1], vert=False, widths=0.42, showfliers=False,
                 patch_artist=True, medianprops=dict(color="white", linewidth=1.6),
                 whiskerprops=dict(color=INK2, linewidth=1),
                 capprops=dict(color=INK2, linewidth=1))
for patch, c in zip(bp["boxes"], [NEUT, S1]):
    patch.set(facecolor=c, edgecolor="none")
for i, s in enumerate(parts):
    axB.text(np.median(s) + 400, i - 0.29, f"{np.median(s):,.0f} bp", fontsize=8.5, color=INK2)
axB.set(yticks=[0, 1], yticklabels=[f"no system\n(n={(~m.carrier).sum():,})",
                                    f"carries a system\n(n={m.carrier.sum():,})"],
        xlim=(0, 20_000))
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel("plasmid size (bp)", fontsize=9, color=INK2)
axB.set_title(f"B · Carriers are {cmp_.loc[True, 'median_bp'] / cmp_.loc[False, 'median_bp']:.1f}× "
              f"larger\nrank-biserial {rbc:+.2f} — a large effect, and a detection bias",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()

## 3. Clonal-redundancy control

The standing control for this project. A pattern carried by one big MOB lineage sequenced many
times is not an ecological signal — the trap that caught the Acinetobacter result in
`cryptic_plasmids.ipynb` cell 10, where a genus-level pattern collapsed to two clusters.

If carriage were a lineage artifact, the big lineages would be the carriers.

In [ ]:
# §3 — the standing clonal-redundancy control for this project. A signal carried by one big
# MOB lineage sequenced many times is not an ecological pattern (cryptic_plasmids.ipynb cell 10).
vc = m.loc[m.carrier, "mob_cluster"].value_counts()
print(f"carriers span {m.loc[m.carrier, 'mob_cluster'].nunique():,} MOB clusters "
      f"of {m['mob_cluster'].nunique():,} present in the searched set")
print(f"  largest cluster {vc.iloc[0]:,} carriers ({vc.iloc[0] / m.carrier.sum() * 100:.1f}%) | "
      f"top 5 {vc.head(5).sum():,} ({vc.head(5).sum() / m.carrier.sum() * 100:.1f}%) | "
      f"{(vc == 1).sum():,} clusters contribute exactly one\n")

cs = m.groupby("mob_cluster").agg(n=("plasmid_id", "size"), carr=("carrier", "sum"))
cs["bin"] = pd.cut(cs.n, [0, 1, 5, 20, 100, 10**7],
                   labels=["1", "2-5", "6-20", "21-100", ">100"])
CLUST = (cs.groupby("bin", observed=True)
         .agg(clusters=("n", "size"), plasmids=("n", "sum"), carriers=("carr", "sum")))
CLUST["rate %"] = CLUST["carriers"] / CLUST["plasmids"] * 100
print("carriage rate by lineage size — the test for clonal inflation:")
print(CLUST.round(2).to_string())
print(f"\n  Carriage FALLS as lineages get bigger "
      f"({CLUST['rate %'].iloc[0]:.1f}% in singleton lineages -> "
      f"{CLUST['rate %'].iloc[-1]:.1f}% in the >100 lineages).")
print("  That is the opposite of clonal inflation: the mega-lineages that dominate this subset")
print("  (AA379 alone holds ~19k plasmids) are defense-POOR. The signal is spread thin across")
print("  many small lineages, so it is not one lineage counted repeatedly.")

DER = m[m.carrier].sample(frac=1, random_state=20260831).drop_duplicates("mob_cluster")
print(f"\ndereplicated to one carrier per lineage: {len(DER):,} plasmids (from {m.carrier.sum():,})")
print("top defense types survive dereplication:")
d1 = DEF.groupby("type")["plasmid_id"].nunique()
d2 = DEF[DEF.plasmid_id.isin(set(DER.plasmid_id))].groupby("type")["plasmid_id"].nunique()
cmpd = pd.DataFrame({"all carriers": d1, "1 per lineage": d2}).fillna(0).astype(int)
cmpd["% retained"] = cmpd["1 per lineage"] / cmpd["all carriers"] * 100
print(cmpd.sort_values("all carriers", ascending=False).head(10).round(0).to_string())

In [ ]:
# Figure 3 — the clonal control. If carriage were a lineage artifact, panel B would slope up.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.6))

top = vc.head(12).iloc[::-1]
y = np.arange(len(top))
axA.barh(y, top.values, height=0.44, color=S1)
for i, v in enumerate(top.values):
    axA.text(v + top.max() * 0.02, i, f"{v:,}", va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=top.index, xlim=(0, top.max() * 1.16))
axA.set_xlabel("carriers in the lineage", fontsize=9, color=INK2)
axA.set_title(f"A · Carriers by MOB lineage — top 12 of "
              f"{m.loc[m.carrier, 'mob_cluster'].nunique():,}\n"
              f"largest holds {vc.iloc[0] / m.carrier.sum() * 100:.0f}% of carriers; "
              f"{(vc == 1).sum():,} lineages contribute one each",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

x = np.arange(len(CLUST))
axB.bar(x, CLUST["rate %"], width=0.5, color=S1)
for i, (r, n) in enumerate(zip(CLUST["rate %"], CLUST["plasmids"])):
    axB.text(i, r + CLUST["rate %"].max() * 0.03, f"{r:.1f}%\nn={n:,}",
             ha="center", fontsize=8, color=INK2)
axB.set(xticks=x, xticklabels=CLUST.index, yticks=[], ylim=(0, CLUST["rate %"].max() * 1.32))
axB.set_xlabel("plasmids in the MOB lineage", fontsize=9, color=INK2)
axB.set_title("B · Carriage falls as lineages grow — not clonal inflation\n"
              "the mega-lineages dominating this subset are defense-poor",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()

## 4. Ecology

One Health compartments are built from `hab_sub` + `is_clinical`, never `hab_top`, with the map
imported directly from `scripts/analyze_amr_onehealth_mobility.py` so this notebook cannot drift
from the Phase-3 analysis.

Compartments differ systematically in plasmid size, so the crude rate is reported alongside a
CDS-stratified version and tested within strata.

In [ ]:
# §4 — ecology. Compartments differ in plasmid size, so the crude rate cannot be read directly;
# it is reported next to a CDS-stratified version, and tested within strata.
mc = m.dropna(subset=["compartment"]).copy()
mc["cds4"] = pd.cut(mc.plasann_n_cds, [0, 5, 8, 12, 10**6],
                    labels=["1-5", "6-8", "9-12", ">12"])
CRUDE = (mc.groupby("compartment")
         .agg(plasmids=("plasmid_id", "size"), carriers=("carrier", "sum")))
CRUDE["rate %"] = CRUDE["carriers"] / CRUDE["plasmids"] * 100
CRUDE = CRUDE.sort_values("rate %", ascending=False)
print(f"assigned to a compartment: {len(mc):,} / {len(m):,} "
      f"({m['compartment'].isna().sum():,} unassigned)\n")
print("crude carriage rate by One Health compartment:")
print(CRUDE.round(2).to_string())

STRAT = mc.pivot_table(index="compartment", columns="cds4", values="carrier",
                       aggfunc="mean", observed=True) * 100
NSTRAT = mc.pivot_table(index="compartment", columns="cds4", values="carrier",
                        aggfunc="size", observed=True)
STRAT = STRAT.loc[CRUDE.index]
NSTRAT = NSTRAT.loc[CRUDE.index]
print("\nsame rates, stratified by CDS count (%):")
print(STRAT.round(1).to_string())

print("\ncompartment x carriage, tested WITHIN each stratum:")
for b, g in mc.groupby("cds4", observed=True):
    ct = pd.crosstab(g["compartment"], g["carrier"])
    if ct.shape[1] == 2 and ct.values.min() >= 1:
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        print(f"  CDS {b:<5} chi2 = {chi2:7.1f}  dof = {dof}  p = {p:.2e}")
LOWEST = STRAT.idxmin()
print("\n  The compartment effect survives stratification in every band, so it is not purely a")
print("  size artefact. The RANKING is not stable across bands, though — lowest-carriage")
print("  compartment per stratum:")
for b, comp in LOWEST.items():
    print(f"    CDS {b:<5} {comp}  ({STRAT.loc[comp, b]:.1f}%)")
print(f"  Human-community is lowest in {(LOWEST == 'Human - community').sum()} of "
      f"{len(LOWEST)} bands and never in the top half.")

In [ ]:
# Figure 4 — ecology, with the size confound held visible rather than corrected away.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 5.0), gridspec_kw={"width_ratios": [1, 1.25]})

c = CRUDE.iloc[::-1]
y = np.arange(len(c))
axA.barh(y, c["rate %"], height=0.44, color=S1)
for i, (r, n) in enumerate(zip(c["rate %"], c["plasmids"])):
    axA.text(r + c["rate %"].max() * 0.02, i, f"{r:.1f}%  (n={n:,})",
             va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=c.index, xlim=(0, c["rate %"].max() * 1.35))
axA.tick_params(axis="y", labelsize=8.5)
axA.set_xlabel("plasmids carrying a defense system (%)", fontsize=9, color=INK2)
axA.set_title("A · Crude carriage by One Health compartment\n"
              "compartments differ in plasmid size, so read panel B before this one",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

im = axB.imshow(STRAT.values, cmap="Blues", aspect="auto", vmin=0)
axB.set(xticks=range(STRAT.shape[1]), xticklabels=STRAT.columns,
        yticks=range(STRAT.shape[0]), yticklabels=STRAT.index)
axB.tick_params(axis="y", labelsize=8.5)
axB.tick_params(axis="x", labelsize=9)
for i in range(STRAT.shape[0]):
    for j in range(STRAT.shape[1]):
        v = STRAT.values[i, j]
        axB.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5,
                 color="white" if v > STRAT.values.max() * 0.55 else INK)
axB.set_xlabel("CDS on the plasmid", fontsize=9, color=INK2)
cb = fig.colorbar(im, ax=axB, pad=0.02)
cb.set_label("carriage rate (%)", fontsize=8, color=INK2)
cb.ax.tick_params(colors=INK2, labelsize=8, length=0)
cb.outline.set_visible(False)
for s in ("top", "right", "left", "bottom"):
    axB.spines[s].set_visible(False)
axB.set_title("B · The same rates, stratified by gene count\n"
              "the gradient runs left→right (size); the compartment effect survives it in\n"
              "every band (χ² p < 1e-11), but the compartment ranking is not stable across bands",
              fontsize=10.5, color=INK, loc="left")

plt.tight_layout()
plt.show()

## 5. What this explains of the dark proteome

In [ ]:
# §5 — what this buys us on the original question: how much of the DARK proteome do defense
# systems explain? Compared against the other screens run on the same 378,552 dark ORFs.
dk = gns[gns.kind.eq("D")]
FP = gns["gene_name"].str.contains("acriia21|ardc", case=False, na=False)
dk_def = int((gns.kind.eq("D") & gns.activity.eq("Defense")).sum())
dk_adf = int((gns.kind.eq("D") & gns.activity.eq("Antidefense") & ~FP).sum())
print(f"dark ORFs assigned to a DefenseFinder system: {len(dk):,} "
      f"({len(dk) / N_DARK * 100:.3f}% of the {N_DARK:,} dark ORFs)")
print(f"  Defense                      {dk_def:>6,}  ({dk_def / N_DARK * 100:.3f}%)")
print(f"  Antidefense (FPs removed)    {dk_adf:>6,}  ({dk_adf / N_DARK * 100:.3f}%)")
print(f"\n  Defense explains ~{dk_def / max(dk_adf, 1):.0f}x more of the dark proteome "
      f"than antidefense.\n")

print("top defense gene families among dark ORFs:")
print(gns[gns.kind.eq("D") & gns.activity.eq("Defense")]
      .groupby("gene_name").size().sort_values(ascending=False).head(10).to_string())

LADDER = pd.Series({
    "dark proteome": N_DARK,
    "Pfam-A can name": 149_160,
    "defense-system genes": dk_def,
    "anti-defense (dbAPIS, adjudicated)": 158,
    "anti-defense (DefenseFinder, FPs removed)": dk_adf,
})
print(f"\nthe dark proteome, by what explains it:")
print(pd.DataFrame({"dark ORFs": LADDER,
                    "% of dark": (LADDER / N_DARK * 100).round(3)}).to_string())
print("\n  Defense systems are a real but small slice. They do not answer what the dark")
print("  plasmidome is; they enlarge the explained fraction from ~39.4% to ~40.7%.")

In [ ]:
# Figure 5 — the result in proportion, on the same log ladder used in antidefense.ipynb so the
# two notebooks are directly comparable.
fig, ax = plt.subplots(figsize=(11, 3.9))
cols = [NEUT, FAINT, S1, S2, S2]
y = np.arange(len(LADDER))[::-1]
ax.barh(y, LADDER.values, height=0.5, color=cols)
for yi, (lab, v) in zip(y, LADDER.items()):
    ax.text(v * 1.15, yi, f"{v:,}   ({v / N_DARK * 100:.3f}%)", va="center",
            fontsize=9, color=INK2)
ax.set(yticks=y, yticklabels=LADDER.index, xscale="log", xlim=(80, N_DARK * 14))
ax.tick_params(axis="y", labelsize=9)
ax.set_xlabel("dark ORFs (log scale)", fontsize=9, color=INK2)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_title("Figure 5 · What explains the dark proteome\n"
             "defense systems (blue) are ~55× larger than anti-defense (orange) — "
             "and still only ~1.3%", fontsize=11, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()

## Conclusion

**Defense systems are real cargo on plasmids the payload definition calls empty.** 3,647 systems on
3,362 plasmids (5.1%), 44% of them restriction-modification, plus `MazEF`, `HEC-09`, `AbiQ`,
`Thoeris`, `Kiwa`, `CBASS` and `Retron`.

**The signal is not clonal.** Carriers span 858 MOB lineages, and carriage *falls* as lineages grow
(10.5% in singleton lineages → 3.0% in lineages >100). The mega-lineages that dominate this subset
are defense-poor, which is the opposite of what clonal inflation would produce.

**There is a real ecological signal**, surviving CDS stratification in all four bands
(χ² p < 1e-11). The compartment ranking is not stable across bands, so the safe claim is that
carriage varies by compartment, not that any one compartment ranks first or last.

**But it does not answer the dark-plasmidome question.** Defense genes account for 5,008 dark ORFs
— **1.32% of the dark proteome**. That is ~55× more than anti-defense manages, and still moves the
explained fraction only from ~39.4% to ~40.7%. The ~57% that neither PlasAnn nor Pfam can name
remains unexplained.

### Limits

- **The size confound is not corrected, only stratified.** Carriage rises 0.69% → 16.68% across CDS
  bands. Every prevalence number here is conditional on that.
- **44% of Defense calls are single-gene.** Correct architecture for some types, incomplete evidence
  for others; the context-supported subset is 2,045 systems on 1,956 plasmids (2.97%).
- **12,077 of 65,805 plasmids have no One Health compartment**, so §4 covers 82% of the set.
- **DefenseFinder models are themselves imperfect** — the AntiDefenseFinder half of this same run
  produced verified false positives (`acriia21` → RepA replication initiators), documented in
  `antidefense.ipynb` §6. No equivalent audit was run on the Defense side; the RM calls are
  multi-gene and therefore better supported, but the single-gene types were not independently
  checked against Pfam.
- **5,569 payload-free plasmids have no PlasAnn genes at all** and are absent from this analysis.